# 8 — P1D and ξ1D of composite overflux fields

Two absorption lines see the **same** absorbers, with line 2 shifted by Δv along the skewer.
- **Optical depths:** τ₁(v) = τ₀ e^{δ(v)} and τ₂(v) = τ₀ e^{δ(v−Δv)}, where δ is a 1D Gaussian random field from `PowerLawGRF` with P(k) = A_s (k/k₀)^{n_s−1}. Every field in this notebook comes from that class.
- **Overfluxes:** `DeltaFieldBuilder` builds the single-line overfluxes δ₁, δ₂ and the composite total

$$\delta_{\rm tot}=\frac{F_1F_2}{\langle F_1F_2\rangle}-1 = w\,(\delta_1+\delta_2+\delta_1\delta_2)-c .$$

- **Statistics:** `OverfluxStatistics1D` measures P1D and ξ1D of the total, of every line-pair term (A, B) and of every order |A|+|B| (notes `sec_flux.tex`).

The shift appears as oscillations in the cross spectra, Re P₁₂ = P₁₁ cos(kΔv), and as peaks of the cross correlations at the line separation. The last section compares every measurement that has an exact expectation with its input. All lengths are in box units (L_box = 100).

In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax
import logging
logging.basicConfig(level=logging.WARNING, format="%(asctime)s - %(levelname)s - %(message)s")

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import skewerjax as sjx
import skewerjax.utils.plotting_tools as pt
from types import SimpleNamespace
from skewerjax.fields.toy_model_samplers import PowerLawGRF
from skewerjax.optical_depth.overflux import DeltaFieldBuilder
from skewerjax.summary_stats.P1D_xi1D import OverfluxStatistics1D
from skewerjax.summary_stats.two_point import xi_from_power
from skewerjax.models import models
from matplotlib.lines import Line2D
jax.config.update("jax_enable_x64", True)   # exact decomposition checks at ~1e-15
plt.style.use("tableau-colorblind10")
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc("font", **font)
plt.rcParams.update(rcnew)
%matplotlib inline
# %matplotlib widget   # interactive figures

## Fields: GRF optical depths
- **Grid:** N_samples skewers of Npix pixels each.
- **Spectrum:** n_s = 0.5 gives a sharp ξ peak, so the shift is easy to see. A_s is set so that σ_δ = 1, which makes the absorption strongly non-linear and the higher orders visible.
- **Shift:** Δv = L_box/10 is an integer number of pixels, so δ₂ is an exact copy of δ₁, and with equal τ₀ the cross spectrum is exactly P₁₂ = P₁₁ e^{ikΔv}.

In [ ]:
seed, N_samples, Npix, L_box = 137, 500, 1000, 100.0
dv = L_box / Npix
v = np.arange(Npix) * dv
delta_v_list = [0.0, L_box / 10]                       # line offsets (box units)
shift = int(round(delta_v_list[1] / dv))               # 100 pixels
tau0 = 1.0

k_0, n_s = 2 * np.pi / 10.0, 0.5
A_s = 1.0 / float(PowerLawGRF(1.0, k_0, n_s, L=L_box, N=Npix).variance())     # sigma_delta = 1
grf = PowerLawGRF(A_s, k_0, n_s, L=L_box, N=Npix)
keys = jax.random.split(jax.random.PRNGKey(seed), N_samples)
delta = jax.vmap(grf.sample)(keys)                                         # (N_samples, Npix)
tau = jnp.stack([tau0 * jnp.exp(delta), tau0 * jnp.exp(jnp.roll(delta, shift, axis=-1))])
overflux = DeltaFieldBuilder(tau, keep_fields=True, keep_field_total=True).build()
labels = overflux.labels                                                    # ["1", "2"]
print(grf, f"  sigma_delta = {np.sqrt(float(grf.variance())):.3f}, <F> = {float(overflux.fields.mean()):.3f}")

idx = np.random.default_rng(seed).choice(N_samples, size=3, replace=False)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
linestyles = ["--", ":", "-."]
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, layout="constrained")
for j, i in enumerate(idx):
    c = colors[j]
    axes[0].plot(v, overflux.field_total[i], lw=1.4, color=c)
    axes[1].plot(v, overflux.total[i], lw=1.4, color=c)
    for m, lab in enumerate(labels):
        axes[0].plot(v, overflux.fields[m, i], lw=2.5, alpha=0.65, color=c, ls=linestyles[m])
        axes[1].plot(v, overflux.delta_terms[lab][i], lw=2.5, alpha=0.65, color=c, ls=linestyles[m])
axes[0].set_ylabel(r"$F$")
axes[1].set(xlabel=r"$v\ [L_{\rm box}/100]$", ylabel=r"$\delta$")
axes[0].legend(handles=[Line2D([], [], color="k", label=r"$F=\prod_j F_j$")]
               + [Line2D([], [], color="k", ls=linestyles[m], label=rf"$F_{lab}$") for m, lab in enumerate(labels)],
               fontsize="small", loc="lower left")
axes[1].legend(handles=[Line2D([], [], color=colors[j], lw=3, label=f"sample #{i}") for j, i in enumerate(idx)],
               fontsize="small", loc="lower left")
plt.show()

## P1D by line pair and by order
1. **Pair terms, one panel per order.** `plot_p1d_by_order` draws the raw pair spectra Re P_AB. The two cross orderings are collapsed into one "sym" curve, and the harmonics of Δv are marked.
2. **Sum of the terms over the total.** `plot_p1d_total_recovery` overlays the total and the exact sum of all weighted pair terms (`Pk_total_from_terms`, including 𝒞), together with the raw order sums.
3. **Orders superimposed.** `plot_orders_superimposed` shows the exact per-order decomposition (`Pk_total_by_order`). Its lower panel is Σ − total, which is rounding error only.

In [ ]:
stats = OverfluxStatistics1D(overflux, dv).compute_p1d().compute_xi1d()
k = np.asarray(stats.k)
k_label = r"$k\ [L_{\rm box}^{-1}]$"

fig, axes, axes_res, _ = pt.plot_p1d_by_order(
    stats.ordered_pairs, stats.Pk_pair, k, delta_v_list, L_box, Npix, tolerance_threshold=1e-3,
    residual_mode="auto_reference", show_harmonics=True, use_symlog_main=False, xlim=(0.0, 8.0), xlabel=k_label)
fig, _, _ = pt.plot_p1d_total_recovery(
    stats.Pk_total, stats.Pk_total_from_terms, stats.Pk_pair, k, delta_v_list, L_box, Npix,
    show_order_sums=True, show_order_sum_residuals=True, main_symlog_linthresh=1e-4, xlabel=k_label)
fig, axs = pt.plot_orders_superimposed(stats, "P", xlabel=k_label, ylabel=r"$P_{\rm 1D}\ [L_{\rm box}]$")
plt.show()

## ξ1D by line pair and by order
These are the same three views for ξ1D, for lags Δv ≥ 0. The box is periodic, so lags beyond L_box/2 repeat. The dashed vertical line marks the line separation Δv₁₂.

In [ ]:
lags = np.asarray(stats.lags)
lag_label = r"$\Delta v / L_{\rm box}$"
dv_norm = [d / L_box for d in delta_v_list]          # these two functions expect lags in units of L_box
fig, axes, axes_res, _, _ = pt.plot_xi_by_order(
    stats.ordered_pairs, stats.xi_pair, lags / L_box, dv_norm, 1.0, use_symlog_main=False, xlabel=lag_label)
fig, _, _, _ = pt.plot_xi_total_recovery(
    stats.xi_total, stats.xi_total_from_terms, stats.xi_pair, lags / L_box, dv_norm, 1.0,
    use_symlog_main=False, xlabel=lag_label)
fig, axs = pt.plot_orders_superimposed(stats, "xi", delta_v_list=delta_v_list, xlabel=r"$\Delta v\ [L_{\rm box}/100]$")
plt.show()

## Input vs recovered
The level of each comparison determines what has to match.

**1. GRF level: exact in expectation.** The estimator P̂ = (dv/N)|rfft δ|² is unbiased for P(k) at every k ≠ 0. ξ̂ = irfft(P̂)/dv is its exact discrete inverse, so the expected ξ is the discrete inverse transform of `grf.power`. The estimators apply no window and no smoothing.
- **Mode counts:** every skewer contributes 2 modes per k (1 at Nyquist), so σ[P̂]/P = √(2/n_modes).
- **Binning:** binned values are mode-weighted means, so they are compared with the input averaged over the same bins with the same weights (`bin_p1d(edges, Pk=…)`).

**2. Flux level, non-linear (the figures above):** F = e^{−τ} and the normalisation by ⟨F⟩ are non-linear, so P[δ_F] ≠ P_in. Only the decomposition identities are exact.

**3. Flux level, linear regime:** a twin GRF with the same k₀ and n_s but a tiny amplitude (σ_δ = 0.01), with τ₁ = τ₀ e^{δ} and τ₂ = a τ₀ e^{δ(v−Δv)}.
- To first order in δ (notes `sec_flux.tex`, Eqs. flux and delta_j), δ_j = F_j/F̄_j − 1 = −τ₀,ⱼ δ_j + O(δ²). Hence P₁₁ → τ₀² P_in.
- For the total, δ_tot → −τ₀[δ(v) + a δ(v−Δv)], hence

$$P_{\rm tot}\to\tau_0^2\,\bigl[1+a^2+2a\cos(k\Delta v)\bigr]\,P_{\rm in}(k),$$

  the McDonald et al. form (`models.mcdonald_factor`; Ma et al. Eq. Pcross_old). ξ_tot is its discrete inverse transform, with peaks of height ∝ a at ±Δv.
- **Corrections:** O(σ_δ²) ≈ 1e-4. The per-skewer mean normalisation of `DeltaFieldBuilder` only removes k = 0.

**4. Decomposition:** Σ orders = Σ weighted terms = total is an algebraic identity, exact to rounding (it is shown in the figures above).

In [ ]:
as_overflux = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": d}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=d)
k_f = 2 * np.pi / L_box
edges = pt.hybrid_k_edges(k_f, grf.k_max[0])                       # one mode per bin up to 12 k_f, then log

# 1. GRF level
st_g = OverfluxStatistics1D(as_overflux(delta), dv).compute_xi1d()
P_g, kb_g, _, nb_g = st_g.bin_p1d(edges)
Pin_g = st_g.bin_p1d(edges, Pk=grf.power(st_g.k))[0]
xi_in_g = np.fft.fftshift(np.asarray(xi_from_power(grf.power(st_g.k), Npix, dv)))
_, xi_err_g = pt.sample_mean_and_error(st_g.xi_total_per_sample)
fig, axs = pt.plot_pk_recovery((P_g, kb_g, edges, nb_g), Pin_g, power=grf.power, label=f"GRF, {N_samples} skewers",
                               k_label=k_label, P_label=r"$P(k)\ [L_{\rm box}]$")
plt.show()
fig, axs = pt.plot_xi_recovery(st_g, xi_in_g, sigma=xi_err_g, max_lag=0.2 * L_box, label="GRF")
axs[1].set_xlabel(r"$\Delta v\ [L_{\rm box}/100]$")
plt.show()

# 3. flux level, linear regime
a_lin = 0.5
grf_lin = PowerLawGRF(1e-4 * A_s, k_0, n_s, L=L_box, N=Npix)                    # sigma_delta = 0.01
delta_lin = jax.vmap(grf_lin.sample)(keys)
tau_lin = jnp.stack([tau0 * jnp.exp(delta_lin), a_lin * tau0 * jnp.exp(jnp.roll(delta_lin, shift, axis=-1))])
st_lin = OverfluxStatistics1D(DeltaFieldBuilder(tau_lin).build(), dv).compute_p1d().compute_xi1d()
k_lin = np.asarray(st_lin.k)
P_tot_exp = tau0 ** 2 * models.mcdonald_factor(k_lin, a_lin, delta_v=delta_v_list[1]) * np.asarray(grf_lin.power(k_lin))
P_11_exp = tau0 ** 2 * np.asarray(grf_lin.power(k_lin))
edges_1 = pt.hybrid_k_edges(k_f, grf.k_max[0], n_lin=Npix // 2, n_log=1)      # one k mode per bin
bins_tot, bins_11 = st_lin.bin_p1d(edges_1), st_lin.bin_p1d(edges_1, Pk=st_lin.Pk_pair[("1", "1")])
Pexp_tot_b, Pexp_11_b = st_lin.bin_p1d(edges_1, Pk=P_tot_exp)[0], st_lin.bin_p1d(edges_1, Pk=P_11_exp)[0]
fig, axs = pt.plot_pk_recovery(bins_11, Pexp_11_b, label=r"$P_{11}$ (line 1 alone)", k_label=k_label,
                               P_label=r"$P\ [L_{\rm box}]$")
pt.plot_pk_recovery(bins_tot, Pexp_tot_b, label=r"$P_{\rm tot}$", ax=axs, k_label=k_label, P_label=r"$P\ [L_{\rm box}]$")
axs[0].plot(k_lin[1:], P_11_exp[1:], color="k", lw=1.5, label=r"$\tau_0^2P_{\rm in}$")
axs[0].plot(k_lin[1:], P_tot_exp[1:], color="k", lw=1.5, ls="--", label=r"$\tau_0^2[1+a^2+2a\cos k\Delta v]P_{\rm in}$")
axs[0].legend(fontsize="x-small", frameon=False)
axs[0].set_xlim(k_f / 2, 8.0)
plt.show()
xi_tot_exp = np.fft.fftshift(np.asarray(xi_from_power(P_tot_exp, Npix, dv)))
_, xi_err_lin = pt.sample_mean_and_error(st_lin.xi_total_per_sample)
fig, axs = pt.plot_xi_recovery(st_lin, xi_tot_exp, sigma=xi_err_lin, label=r"$\xi_{\rm tot}$, linear regime")
for a_ in axs:
    a_.axvline(delta_v_list[1], color="C3", ls="--", lw=1); a_.axvline(-delta_v_list[1], color="C3", ls="--", lw=1)
axs[1].set_xlabel(r"$\Delta v\ [L_{\rm box}/100]$")
plt.show()

## Checks

In [ ]:
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append((name, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")
rel = lambda a, b: float(np.max(np.abs(np.asarray(a) - np.asarray(b))) / np.max(np.abs(np.asarray(b))))
def chi2_check(name, binned, P_exp, n_real=1, sigma=None):
    P_b = np.asarray(binned[0]); sig = pt.pk_sigma(P_exp, binned[3], n_real) if sigma is None else sigma
    pull = (P_b - np.asarray(P_exp)) / sig; pull = pull[np.isfinite(pull)]
    check(name, np.mean(pull ** 2) < 1.5 and np.max(np.abs(pull)) < 5, pt.recovery_summary(P_b, P_exp, sig, "P"))
def xi_check(name, xi, xi_exp, sig):
    pull = (np.asarray(xi) - xi_exp) / sig
    check(name, np.mean(np.abs(pull) < 2) > 0.8 and np.max(np.abs(pull)) < 6, pt.recovery_summary(xi, xi_exp, sig, "xi", ratio=False))

chi2_check("GRF level: P(k) = input", (P_g, kb_g, edges, nb_g), Pin_g)
xi_check("GRF level: xi = discrete inverse transform of the input", st_g.xi_total, xi_in_g, xi_err_g)
chi2_check("flux level, linear: P_11 -> tau0^2 P_in", bins_11, Pexp_11_b)
chi2_check("flux level, linear: P_tot -> tau0^2 (1 + a^2 + 2a cos k dv) P_in", bins_tot, Pexp_tot_b)
xi_check("flux level, linear: xi_tot = inverse transform of the expected P_tot", st_lin.xi_total, xi_tot_exp, xi_err_lin)
P_terms = sum(np.asarray(stats.Pk_total_from_terms_contributions[p]) for p in stats.term_pairs)
xi_terms = sum(np.asarray(stats.xi_total_from_terms_contributions[p]) for p in stats.term_pairs)
check("decomposition, P1D: sum of orders = sum of terms = total",
      max(rel(sum(stats.Pk_total_by_order.values()), stats.Pk_total), rel(P_terms, stats.Pk_total)) < 1e-10,
      f"rel {rel(sum(stats.Pk_total_by_order.values()), stats.Pk_total):.1e}, {rel(P_terms, stats.Pk_total):.1e}")
check("decomposition, xi1D: sum of orders = sum of terms = total",
      max(rel(sum(stats.xi_total_by_order.values()), stats.xi_total), rel(xi_terms, stats.xi_total)) < 1e-10,
      f"rel {rel(sum(stats.xi_total_by_order.values()), stats.xi_total):.1e}, {rel(xi_terms, stats.xi_total):.1e}")
dv12 = delta_v_list[1] - delta_v_list[0]
P12 = np.asarray(stats.Pk_pair_complex[("1", "2")]); P11 = np.asarray(stats.Pk_pair[("1", "1")])
check("shift signature: P_12 = P_11 exp(+i k dv12) (exact, non-linear case)", rel(P12, P11 * np.exp(1j * k * dv12)) < 1e-10,
      f"rel {rel(P12, P11 * np.exp(1j * k * dv12)):.1e}")
xi12 = np.fft.fftshift(np.asarray(stats.xi_pair[("1", "2")]))
lag_peak = lags[np.argmax(xi12)]
check("shift signature: xi_12 peaks at +dv12", abs(lag_peak - dv12) <= dv, f"peak at {lag_peak:.2f}, dv12 = {dv12:.2f}")
print(f"\nSUMMARY: {sum(p for _, p in CHECKS)}/{len(CHECKS)} checks passed")